# 🔬 Feature Extraction – Tối ưu hóa Features

Mục tiêu: Xây dựng bộ features **có khả năng phân biệt cao** giữa các action:
- `-1` → apply integral trực tiếp (basic: const, var đơn, sin/cos đơn, sqrt đơn)
- `0` → unknown / cần phân tích thêm
- `2` → expand (có `(expr)^n` với n>=2)
- `3` → u-substitution (trig với inner ≠ x, sqrt với inner ≠ x)
- `4` → frac rule (frac với tử là bội của đạo hàm mẫu)
- `7` → simplify/power rule nâng cao
- `8` → basic tổng hợp (add/sub của các hàm đơn)
- `9` → sqrt rule (sqrt đơn giản)
- `10` → compound (tổ hợp phức tạp)

In [1]:
import pandas as pd
import sys
import os
import re

sys.path.append(os.path.abspath("../.."))

from ai.utils.integral import Integral
from ai.utils.expr.expr_node import ExprNode
from ai.utils.expr.trig.expr_sin import SinExprNode
from ai.utils.expr.trig.expr_cos import CosExprNode
from ai.utils.expr.trig.expr_tan import TanExprNode
from ai.utils.expr.operation.expr_add import AddExprNode
from ai.utils.expr.operation.expr_sub import SubExprNode
from ai.utils.expr.operation.expr_mul import MulExprNode
from ai.utils.expr.operation.expr_frac import FracExprNode
from ai.utils.expr.value.expr_var import VarExprNode
from ai.utils.expr.value.expr_const import ConstExprNode
from ai.utils.expr.Power.expr_mono import MonoExprNode
from ai.utils.expr.expr_log import LogExprNode
from ai.utils.expr.Power.expr_sqrt import SqrtExprNode
from ai.utils.expr.Power.expr_power import PowerExprNode

from ai.model.feature_extractor import extract_features
print('✅ Imports OK')

✅ Imports OK


In [2]:
df_clean = pd.read_csv("../data/processed/dataset.csv")
df_clean = df_clean.head(250)
print(f'📦 Dataset: {len(df_clean)} mẫu')
print(df_clean.head())

📦 Dataset: 250 mẫu
                               integrand  action
0  \int_{0}^{1}\frac{2*x}{2*x}+{x}^{4}dx       1
1            \int_{0}^{1}(x+1)*(x+2)+xdx       1
2                      \int_{0}^{1}2*xdx       1
3                        \int_{0}^{1}8dx       0
4    \int_{0}^{1}\frac{1}{x}-x+\sin{x}dx       1


In [3]:
# ─── Helper functions ────────────────────────────────────────────────────────

def is_pure_var_or_const(node):
    return isinstance(node, (VarExprNode, ConstExprNode))

def is_linear_expr(node):
    """Node là biểu thức tuyến tính: ax, ax+b, ax-b, x+b..."""
    if isinstance(node, (VarExprNode, ConstExprNode)):
        return True
    if isinstance(node, MulExprNode):
        # c*x dạng
        if isinstance(node.left, ConstExprNode) and isinstance(node.right, VarExprNode):
            return True
        if isinstance(node.right, ConstExprNode) and isinstance(node.left, VarExprNode):
            return True
    if isinstance(node, (AddExprNode, SubExprNode)):
        return is_linear_expr(node.left) and is_linear_expr(node.right)
    return False

def is_monomial(node):
    """Node là đơn thức: x^n hoặc c*x^n"""
    if isinstance(node, (MonoExprNode, PowerExprNode)):
        return True
    if isinstance(node, MulExprNode):
        if isinstance(node.left, ConstExprNode) and isinstance(node.right, (MonoExprNode, PowerExprNode, VarExprNode)):
            return True
        if isinstance(node.right, ConstExprNode) and isinstance(node.left, (MonoExprNode, PowerExprNode, VarExprNode)):
            return True
    return False

def has_non_trivial_inner(trig_or_sqrt_node):
    """Hàm lượng giác / sqrt có inner phức tạp hơn x đơn?"""
    inner = trig_or_sqrt_node.left
    if inner is None:
        return False
    if isinstance(inner, VarExprNode):
        return False  # trivial: sin(x)
    return True      # non-trivial: sin(2x), cos(x+1)...

def count_nodes(node):
    """Đếm tổng số node trong cây"""
    if node is None:
        return 0
    count = 1
    if hasattr(node, 'left') and node.left is not None:
        count += count_nodes(node.left)
    if hasattr(node, 'right') and node.right is not None:
        count += count_nodes(node.right)
    return count

def get_depth(node, d=0):
    """Độ sâu tối đa của cây"""
    if node is None:
        return d
    dl = d
    dr = d
    if hasattr(node, 'left') and node.left is not None:
        dl = get_depth(node.left, d+1)
    if hasattr(node, 'right') and node.right is not None:
        dr = get_depth(node.right, d+1)
    return max(dl, dr)

print('✅ Helper functions defined')

✅ Helper functions defined


In [4]:
# ─── extract_features đã được chuyển sang feature_extractor.py ─────────────
# Xem: ai/model/feature_extractor.py
#
# Ba nhóm features tối ưu mới:
#   NHÓM 1 – detect_root_add_sub()      → phát hiện root ADD/SUB (action 8)
#   NHÓM 2 – detect_const_times_func()  → phát hiện c*f(x)
#   NHÓM 3 – detect_basic_antiderivative() → nguyên hàm cơ bản 1 bước (action -1)

# ── Smoke test ──────────────────────────────────────────────────────────────
test_cases = [
    (r'\int_{0}^{1}x+\sin{x}dx',        'action 8 – root ADD'),
    (r'\int_{0}^{1}2*\sin{x}dx',         'action -1 – const*trig'),
    (r'\int_{0}^{1}{x}^{3}dx',            'action -1 – mono'),
    (r'\int_{0}^{1}\cos{2*x}dx',          'action 3 – trig nontrivial'),
    (r'\int_{0}^{1}{x+1}^{2}dx',          'action 2 – expand'),
]

print('\n=== Smoke Tests ===')
for latex, desc in test_cases:
    f = extract_features(latex)
    if f is None:
        print(f'  [{desc}] PARSE ERROR')
        continue
    key_flags = {
        'root_is_add_or_sub' : f.get('root_is_add_or_sub', 0),
        'root_const_times_func': f.get('root_const_times_func', 0),
        'one_step_basic'       : f.get('one_step_basic', 0),
        'mono_with_add_inner'  : f.get('mono_with_add_inner', 0),
        'trig_nontrivial_inner': f.get('trig_nontrivial_inner', 0),
    }
    print(f'  [{desc}]')
    for k, v in key_flags.items():
        print(f'    {k}: {v}')



=== Smoke Tests ===
  [action 8 – root ADD]
    root_is_add_or_sub: 1
    root_const_times_func: 0
    one_step_basic: 0
    mono_with_add_inner: 0
    trig_nontrivial_inner: 0
  [action -1 – const*trig]
    root_is_add_or_sub: 0
    root_const_times_func: 1
    one_step_basic: 0
    mono_with_add_inner: 0
    trig_nontrivial_inner: 0
  [action -1 – mono]
    root_is_add_or_sub: 0
    root_const_times_func: 0
    one_step_basic: 1
    mono_with_add_inner: 0
    trig_nontrivial_inner: 0
  [action 3 – trig nontrivial]
    root_is_add_or_sub: 0
    root_const_times_func: 0
    one_step_basic: 0
    mono_with_add_inner: 0
    trig_nontrivial_inner: 1
  [action 2 – expand]
    root_is_add_or_sub: 0
    root_const_times_func: 0
    one_step_basic: 1
    mono_with_add_inner: 1
    trig_nontrivial_inner: 0


In [5]:
features = df_clean["integrand"].apply(extract_features)
features = features.dropna()
df = pd.DataFrame(features.tolist())

# Thêm label
df["label"] = df_clean.loc[features.index, "action"]
df["label"] = df["label"].fillna(0).astype(int)

print(f'✅ Shape: {df.shape}')
print(f'📐 Số features: {df.shape[1] - 1}')
print(f'🏷️  Classes: {sorted(df["label"].unique())}')
print(df.head())

✅ Shape: (248, 87)
📐 Số features: 86
🏷️  Classes: [0, 1, 2, 3, 4, 6, 7, 8]
   root_is_frac  root_is_trig  root_is_log  root_is_var  root_is_const  \
0             0             0            0            0              0   
1             0             0            0            0              0   
2             0             0            0            0              0   
3             0             0            0            0              1   
4             0             0            0            0              0   

   has_trig  has_sin  has_cos  has_tan  has_sqrt  ...  mul_sqrt_other  \
0         0        0        0        0         0  ...               0   
1         0        0        0        0         0  ...               0   
2         0        0        0        0         0  ...               0   
3         0        0        0        0         0  ...               0   
4         1        1        0        0         0  ...               0   

   frac_and_trig  frac_and_sqrt  add_and_

In [6]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Phân phối nhãn
cnt = Counter(df['label'])
print('📊 Phân phối nhãn sau khi extract features:')
for k in sorted(cnt):
    print(f'  action={k:3d}: {cnt[k]} ({cnt[k]/len(df)*100:.1f}%)')

# Correlation của từng feature với label
X = df.drop(columns=['label'])
y = df['label']
print(f'\n📐 Feature list ({len(X.columns)} features):')
for c in X.columns:
    print(f'  {c}')

📊 Phân phối nhãn sau khi extract features:
  action=  0: 56 (22.6%)
  action=  1: 53 (21.4%)
  action=  2: 2 (0.8%)
  action=  3: 17 (6.9%)
  action=  4: 1 (0.4%)
  action=  6: 15 (6.0%)
  action=  7: 46 (18.5%)
  action=  8: 58 (23.4%)

📐 Feature list (86 features):
  root_is_frac
  root_is_trig
  root_is_log
  root_is_var
  root_is_const
  has_trig
  has_sin
  has_cos
  has_tan
  has_sqrt
  has_power
  has_mono
  has_frac
  has_exp
  has_log
  count_sin
  count_cos
  count_tan
  count_frac
  count_sqrt
  count_pow
  count_mono
  count_log
  count_exp
  count_x
  count_const
  tree_depth
  tree_nodes
  root_is_add
  root_is_sub
  root_is_add_or_sub
  root_add_both_simple
  has_add
  has_sub
  count_add
  count_sub
  count_terms
  root_is_mul
  root_const_times_func
  is_mul_const_var
  root_const_times_trig
  root_const_times_sqrt
  root_const_times_frac
  any_const_times_func
  has_mul
  count_mul
  is_pure_var
  is_pure_const
  root_is_mono
  is_single_trig_simple
  root_is_sqrt
  r

In [7]:
# Quick test: feature variance per class
print('🔍 Mean feature values per action (selected features):')
key_feats = [
    'root_is_add', 'root_is_frac', 'root_is_trig', 'root_is_sqrt', 'root_is_mono',
    'root_is_var', 'mono_with_add_inner', 'trig_nontrivial_inner',
    'sqrt_nontrivial_inner', 'frac_linear_over_linear', 'root_is_sqrt_simple',
    'root_is_add_or_sub', 'tree_depth', 'total_ops', 'is_simple'
]
print(df.groupby('label')[key_feats].mean().round(2).to_string())

🔍 Mean feature values per action (selected features):
       root_is_add  root_is_frac  root_is_trig  root_is_sqrt  root_is_mono  root_is_var  mono_with_add_inner  trig_nontrivial_inner  sqrt_nontrivial_inner  frac_linear_over_linear  root_is_sqrt_simple  root_is_add_or_sub  tree_depth  total_ops  is_simple
label                                                                                                                                                                                                                                                         
0             0.18          0.05          0.21          0.30          0.18          0.0                 0.09                   0.18                   0.29                     0.02                 0.25                0.18        2.84       1.88       0.34
1             0.42          0.09          0.11          0.09          0.17          0.0                 0.00                   0.11                   0.15                     0.06  

In [8]:
df.to_csv("../data/processed/integral_dataset.csv", index=False)
print(f'💾 Đã lưu integral_dataset.csv ({df.shape[0]} rows, {df.shape[1]} cols)')
print(f'   Features: {list(df.columns[:-1])}')

💾 Đã lưu integral_dataset.csv (248 rows, 87 cols)
   Features: ['root_is_frac', 'root_is_trig', 'root_is_log', 'root_is_var', 'root_is_const', 'has_trig', 'has_sin', 'has_cos', 'has_tan', 'has_sqrt', 'has_power', 'has_mono', 'has_frac', 'has_exp', 'has_log', 'count_sin', 'count_cos', 'count_tan', 'count_frac', 'count_sqrt', 'count_pow', 'count_mono', 'count_log', 'count_exp', 'count_x', 'count_const', 'tree_depth', 'tree_nodes', 'root_is_add', 'root_is_sub', 'root_is_add_or_sub', 'root_add_both_simple', 'has_add', 'has_sub', 'count_add', 'count_sub', 'count_terms', 'root_is_mul', 'root_const_times_func', 'is_mul_const_var', 'root_const_times_trig', 'root_const_times_sqrt', 'root_const_times_frac', 'any_const_times_func', 'has_mul', 'count_mul', 'is_pure_var', 'is_pure_const', 'root_is_mono', 'is_single_trig_simple', 'root_is_sqrt', 'root_is_sqrt_simple', 'root_is_inv_x', 'root_const_times_mono', 'root_const_times_inv_x', 'one_step_basic', 'count_nontrivial_funcs', 'count_distinct_func_